<a href="https://colab.research.google.com/github/nesrintekbas02-cmd/nesrinNDVI/blob/main/nesrinver2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q geemap earthengine-api

In [ ]:
import ee
import geemap

try:
    ee.Authenticate()
    proje_id = input("Lütfen Google Cloud Proje ID'nizi girin (Örn: kanaryalar): ")
    ee.Initialize(project=proje_id)
    print("✅ Earth Engine Başarıyla Başlatıldı!")
except Exception as e:
    print(f"❌ Hata: {e}")

In [ ]:
# @title
# Kahramanmaraş koordinatları
k_maras_poi = ee.Geometry.Point([36.92, 37.57])

# Sentinel-2 verilerini filtreleme (Yine bulutsuz yaz ayları)
s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterBounds(k_maras_poi) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))

# 1. Deprem Öncesi (2022) ve Sonrası (2023) Görüntüleri
oncesi_img = s2.filterDate('2022-06-01', '2022-09-01').median()
sonrasi_img = s2.filterDate('2023-06-01', '2023-09-01').median()

# 2. NDVI Hesaplama (B8: NIR, B4: Red)
ndvi_oncesi = oncesi_img.normalizedDifference(['B8', 'B4'])
ndvi_sonrasi = sonrasi_img.normalizedDifference(['B8', 'B4'])

# 3. Görselleştirme Ayarları (Kırmızı-Sarı-Yeşil Paleti)
# -1 ile 0 arası: Su/Yapı (Kırmızımsı)
# 0.2 - 0.5: Zayıf bitki (Sarı)
# 0.5 - 1.0: Yoğun bitki (Koyu Yeşil)
ndvi_viz = {
    'min': 0,
    'max': 0.8,
    'palette': ['#d73027', '#f46d43', '#fdae61', '#fee08b', '#d9ef8b', '#a6d96a', '#66bd63', '#1a9850']
}

# 4. Haritayı Oluştur
m = geemap.Map(center=[37.57, 36.92], zoom=13)

# Katmanları hazırla
sol_ndvi = geemap.ee_tile_layer(ndvi_oncesi, ndvi_viz, '2022 NDVI (Öncesi)')
sag_ndvi = geemap.ee_tile_layer(ndvi_sonrasi, ndvi_viz, '2023 NDVI (Sonrası)')

# Kaydırmalı (Split-panel) haritayı ekrana bas
m.split_map(left_layer=sol_ndvi, right_layer=sag_ndvi)

# Haritaya renk skalası (legend) ekleyelim ki jüri neye baktığını anlasın
m.add_colorbar(ndvi_viz, label="NDVI Değeri", orientation="horizontal", layer_name="NDVI")

m